# Groupwise repeated-structure reconstruction

## Goal checkpoint

Test whether a shared smooth latent PT atlas fitted to both healthy specimens contributes beyond matched pooled DPT and scFates curves when the second specimen supplies only unlabeled calibration profiles. The intended object is one ordered distribution path from repeated cross-sections; nephron identities remain unknown and are integrated out.

For each source direction and gene fold, half of query structures are selected by a preregistered structure-ID hash for calibration and the complement is evaluated. Full and S3-only regimes use that same evaluation half. In the S3-only stress regime, labels select calibration rows, then every query calibration anatomy value sent to the models is -1.

Source anatomy anchors graph roots and orientation; joint models also use source ordinal likelihoods, so supervision differs. Joint posterior concentration is not calibrated anatomical certainty. The fitted path and specimen intercept summarize integrated variation; they do not reconstruct individual nephrons or identify injury directions. All-gene exposure can create compositional dependence. No query trajectories or depth enter reconstruction. This healthy pilot does not support AKI, human, regulator, or tuning branches.


## Frozen protocol

Compare three Gaussian-atlas special cases on identical pooled source plus calibration profiles and pooled scFates initialization: specimen-balanced with constant offsets; specimen-balanced without offsets; observation-weighted without offsets. Use grid 101, roughness 0.001, offset penalty 4, anatomy strength 1, anatomy width 0.1, tolerance 1e-5, and iteration cap 60. Retain solver histories and nonconvergence; a nonconverged arm is marked failed and contributes no biological prediction while other arms continue.

Use one 15-neighbor extension for each fitted axis and source-only response-gene decoders. Report native expression-only atlas projection separately, with the fitted target intercept frozen. Source-only DPT/scFates provide zero-calibration context; pooled graph baselines control for the extra unlabeled profiles. Retain fixed source-selected within-segment programs, a quadratic segment-plus-coverage oracle, a segment-only mean oracle, and paired quarter-read perturbations. The two oracles use query labels as privileged comparators. The quadratic coverage oracle reuses notebook-24 exposure, spot-count, and detected-gene covariates; these do not enter trajectory fitting or method selection.

The Gaussian likelihood integrates over a uniform latent grid mixture. It does not estimate specimen-specific sampling frequencies, and grid spacing is not physical length. Partial coverage tests this likelihood assumption as well as shared curve fitting.


In [ ]:
from pathlib import Path
import hashlib, io, json, sys, os, argparse
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.joint_repeated_atlas import fit_joint_atlas,project_joint_atlas
from pseudospace.trajectory_benchmark import fit_scfates_axis,fit_dpt_axis,project_neighbor_axis
from pseudospace.within_segment_validation import fit_segment_covariate_oracle,predict_segment_covariate_oracle
from pseudospace.stage_cache import cached_payload
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'groupwise_repeated_structure_reconstruction'
OUT.mkdir(parents=True,exist_ok=True)
REFERENCE_OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
NOTEBOOK_LOGIC_VERSION='25.groupwise.2'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
CONTROLS=('Ctrl1A2','Ctrl1A4')
GRID=np.linspace(0,1,101)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
JOINT_PARAMS=dict(grid=GRID,roughness=.001,offset_penalty=4.,anatomy_strength=1.,
                  anatomy_width=.1,max_iter=60,tol=1e-5,allow_unlabeled_anatomy=True)
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py',
 'trajectory_benchmark.py','repeated_structure.py','joint_repeated_atlas.py',
 'within_segment_validation.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
BASELINE_SCORE_PATH=REFERENCE_OUT/'scores.csv'
BASELINE_POSITION_PATH=REFERENCE_OUT/'positions.csv'
if not BASELINE_SCORE_PATH.is_file() or not BASELINE_POSITION_PATH.is_file():
    raise FileNotFoundError('Required notebook-21 score and position baselines are missing')
BASELINE_SCORE_DIGEST=hashlib.sha256(BASELINE_SCORE_PATH.read_bytes()).hexdigest()
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
PANEL_DIGEST=hashlib.sha256(PANEL_PATH.read_bytes()).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,components=N_COMPONENTS,
 folds=N_FOLDS,markers=sorted(MARKERS),grid=GRID.tolist(),count_representation_theta=100.,
 calibration_hash='sha256(structure_id) first8 % 2 == 0',evaluation_hash='complement',
 regimes=['full','S3_only'],graph={'dpt_neighbors':30,'scfates_nodes':30,'anchor_mask':'source-only'},
 joint={**JOINT_PARAMS,'grid':GRID.tolist(),
  'arms':['joint_balanced_offset','joint_balanced_no_offset','joint_pooled_no_offset']},
 projection_neighbors=15,gene_spline_ridge=.001,quarter_read_fraction=.25,
 quarter_read_seed_rule='seed+fold',oracle_ridge=.01,oracle_degree=2,
 baseline_digest=BASELINE_SCORE_DIGEST,panel_digest=PANEL_DIGEST,versions=VERSIONS)
reference_scores=pd.read_csv(BASELINE_SCORE_PATH)
reference_positions=pd.read_csv(BASELINE_POSITION_PATH)
reference_positions=reference_positions[reference_positions.method.isin(['scFates_neighbor','DPT_neighbor'])]
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','test_specimen','fold','segment','gene']).any():
    raise ValueError('Duplicate frozen panel genes')
print('Installed benchmark versions:',VERSIONS)


## Actual healthy inputs

Use validated repeated-control inputs and three deterministic gene folds. For each fold, modeling genes exclude markers and response-fold genes individually. Fit the count transform on source raw counts and all-gene library exposure, then freeze it for query profiles. No query trajectory or depth enters reconstruction.


In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly two healthy controls')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']
X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0):
    raise ValueError('Invalid expression or all-gene exposure')
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,
                code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))


In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0)
                               for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,prediction in predictions.items():
        scaled=(response-prediction)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any():
                out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),
                                normalized_mse=float(scaled[use].mean())))
    return out

def selected_rows(response,predictions,train_response,train_anatomy,eval_anatomy,panel,gene_to_j,meta):
    out=[]
    for seg,name in enumerate(['S1','S2','S3']):
        selected=panel[panel.segment==name]
        if selected.empty or not set(selected.gene).issubset(gene_to_j):
            raise ValueError('Invalid frozen panel for '+name)
        jj=np.array([gene_to_j[g] for g in selected.gene],int)
        variance=np.maximum(np.var(train_response[train_anatomy==seg],axis=0),1e-3)[jj]
        use=eval_anatomy==seg
        if not use.any(): raise ValueError('Missing evaluation segment '+name)
        for method,prediction in predictions.items():
            out.append(dict(**meta,method=method,segment=name+'_selected',n=int(use.sum()),
              selected_gene_count=len(jj),normalized_mse=float(np.mean(
               (response[use][:,jj]-prediction[use][:,jj])**2/variance))))
    return out


## Groupwise fits and matched comparisons

Hash halves remain disjoint for both calibration regimes. Query labels only select S3 stress calibration rows and support the named oracle/stratified scores. Every model receives unknown query anatomy. Source-only and pooled graph axes use the same anchor mask; joint ablations share pooled scFates initialization. A failed joint arm keeps its fit history and diagnostics but emits no predictions.


In [ ]:
def run_groupwise():
    scores,selected_scores,thin_scores,stability,positions,diagnostics,histories,failures,sourcefolds,graphs=([],[],[],[],[],[],[],[],[],[])
    arms=[('joint_balanced_offset',True,True),('joint_balanced_no_offset',True,False),
          ('joint_pooled_no_offset',False,False)]
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        query_ids=ids[te]
        calibration=np.array([int(hashlib.sha256(x.encode()).hexdigest()[:8],16)%2==0 for x in query_ids])
        evaluation=~calibration
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold)); eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                C=dense_panel(raw_counts,tr,mg); Q=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(C,raw_total[tr],specimens[tr],N_COMPONENTS,theta=100.,seed=SEED)
                qY=transform_count_representation(Q,raw_total[te],transform)
                sa=anatomy[tr]; response=X[tr][:,eg]; qresponse=X[te][:,eg]
                variance=normalized_variance(response,specimens[tr])
                sz,sm=fit_scfates_axis(Y,sa,nodes=30,seed=SEED)
                dz,dm=fit_dpt_axis(Y,sa,n_neighbors=30,seed=SEED)
                sdec=fit_gene_spline(response,sz,specimens[tr])
                ddec=fit_gene_spline(response,dz,specimens[tr])
                source_z={'source_scFates_neighbor':project_neighbor_axis(Y,sz,qY,k=15),
                          'source_DPT_neighbor':project_neighbor_axis(Y,dz,qY,k=15)}
                reproduced=pd.DataFrame(primary_rows(qresponse,{
                    'scFates_neighbor':sdec.predict(source_z['source_scFates_neighbor']),
                    'DPT_neighbor':ddec.predict(source_z['source_DPT_neighbor'])},
                    variance,anatomy[te],key))
                prior=reference_scores.query('source_specimen==@source and fold==@fold and method in ["scFates_neighbor","DPT_neighbor"]')
                joined=reproduced.merge(prior,on=['method','segment'],suffixes=('_new','_old'),validate='one_to_one')
                if len(joined)!=8 or not np.allclose(joined.normalized_mse_new,joined.normalized_mse_old,rtol=1e-8,atol=1e-10):
                    raise ValueError('Notebook-21 source reference scores did not reproduce')
                for method,zq in source_z.items():
                    refmethod='scFates_neighbor' if 'scFates' in method else 'DPT_neighbor'
                    saved=reference_positions.query('source_specimen==@source and fold==@fold and method==@refmethod').set_index('structure_id')
                    if saved.index.duplicated().any() or set(saved.index)!=set(query_ids):
                        raise ValueError('Notebook-21 query structure IDs mismatch')
                    if not np.allclose(zq,saved.loc[query_ids,'z'],rtol=1e-8,atol=1e-10):
                        raise ValueError('Notebook-21 source coordinates did not reproduce')

                # Coverage oracle follows the existing privileged diagnostic covariates.
                detected_train=np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel()
                detected_query=np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel()
                train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(detected_train)])
                query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(detected_query)])
                oracle=fit_segment_covariate_oracle(response,train_cov,sa,specimens[tr],ridge=.01,degree=2)
                eval_response=qresponse[evaluation]; eval_a=anatomy[te][evaluation]; eval_ids=query_ids[evaluation]
                oracle_pred=predict_segment_covariate_oracle(oracle,query_cov[evaluation],eval_a)
                means={a:response[sa==a].mean(axis=0) for a in (0,1,2)}
                segment_pred=np.vstack([means[int(a)] for a in eval_a])
                panel=frozen_panel.query('source_specimen==@source and test_specimen==@target and fold==@fold')
                gene_to_j={g:j for j,g in enumerate(genes[eg])}
                full=sparse.csr_matrix(raw_counts[te]); rng=np.random.default_rng(SEED+fold)
                thin=sparse.csr_matrix((rng.binomial(full.data.astype(np.int64),.25),
                    full.indices.copy(),full.indptr.copy()),shape=full.shape)
                thinlib=np.asarray(thin.sum(axis=1)).ravel().astype(float)
                if np.any(thinlib<=0): raise ValueError('Quarter-read thinning produced zero exposure')
                thin_qY=transform_count_representation(
                    dense_panel(thin,np.ones(len(thinlib),bool),mg),thinlib,transform)[evaluation]
                sourcefolds.append(dict(**key,status='complete',calibration_n=int(calibration.sum()),
                                        evaluation_n=int(evaluation.sum())))
            except Exception as exc:
                failures.append(dict(**key,regime='',method='sourcefold_setup',error=str(exc)))
                sourcefolds.append(dict(**key,status='failed',error=str(exc)))
                continue

            for regime in ('full','S3_only'):
                cal=calibration if regime=='full' else calibration&(anatomy[te]==2)
                if not cal.any():
                    failures.append(dict(**key,regime=regime,method='calibration_population',error='empty'))
                    continue
                P=np.vstack([Y,qY[cal]])
                ps=np.r_[specimens[tr],np.repeat(target,int(cal.sum()))]
                anchor=np.r_[np.ones(len(Y),bool),np.zeros(int(cal.sum()),bool)]
                pa=np.r_[sa,np.full(int(cal.sum()),-1,dtype=int)]
                try:
                    pz_d,md=fit_dpt_axis(P,pa,anchor_mask=anchor,n_neighbors=30,seed=SEED)
                    pz_s,ms=fit_scfates_axis(P,pa,anchor_mask=anchor,nodes=30,seed=SEED)
                    graphs.extend([dict(**key,regime=regime,method='pooled_DPT',status='complete',metadata=json.dumps(md)),
                                   dict(**key,regime=regime,method='pooled_scFates',status='complete',metadata=json.dumps(ms))])
                except Exception as exc:
                    graphs.append(dict(**key,regime=regime,method='pooled_graphs',status='failed',metadata=str(exc)))
                    failures.append(dict(**key,regime=regime,method='pooled_graphs',error=str(exc)))
                    continue
                nsrc=len(Y); evalY=qY[evaluation]
                coords={}; thincoords={}; decoders={}; entropy={}
                decoders['source_scFates_neighbor']=sdec
                decoders['source_DPT_neighbor']=ddec
                decoders['pooled_scFates_neighbor']=fit_gene_spline(response,pz_s[:nsrc],specimens[tr])
                decoders['pooled_DPT_neighbor']=fit_gene_spline(response,pz_d[:nsrc],specimens[tr])
                for name,train,axis in [('source_scFates_neighbor',Y,sz),('source_DPT_neighbor',Y,dz),
                                        ('pooled_scFates_neighbor',P,pz_s),('pooled_DPT_neighbor',P,pz_d)]:
                    coords[name]=project_neighbor_axis(train,axis,evalY,k=15)
                    thincoords[name]=project_neighbor_axis(train,axis,thin_qY,k=15)
                for name,balance,offsets in arms:
                    try:
                        atlas=fit_joint_atlas(P,ps,pa,initial_z=pz_s,balance_specimens=balance,
                                              fit_offsets=offsets,**JOINT_PARAMS)
                        converged=bool(atlas['converged'])
                        target_offset=atlas['offsets'].get(target,np.zeros(P.shape[1]))
                        info=dict(**key,regime=regime,method=name,converged=converged,
                          iterations=len(atlas['history']),sigma2=float(atlas['sigma2']),
                          cutpoints=json.dumps(atlas['cutpoints'].tolist()),
                          offsets=json.dumps({s:float(np.linalg.norm(v)) for s,v in atlas['offsets'].items()}),
                          source_latent_vs_pooled_sc_spearman=float(
                              spearmanr(atlas['z_mean'][:nsrc],pz_s[:nsrc]).statistic))
                        diagnostics.append(info)
                        for record in atlas['history']:
                            histories.append(dict(**key,regime=regime,method=name,**record))
                        if not converged:
                            failures.append(dict(**key,regime=regime,method=name,
                                error='nonconverged; biological prediction skipped'))
                            continue
                        trainz=atlas['z_mean'][:nsrc]
                        decoder=fit_gene_spline(response,trainz,specimens[tr])
                        neighbor_z=project_neighbor_axis(P,atlas['z_mean'],evalY,k=15)
                        neighbor_thin=project_neighbor_axis(P,atlas['z_mean'],thin_qY,k=15)
                        native=project_joint_atlas(atlas,evalY,offset=target_offset)
                        native_thin=project_joint_atlas(atlas,thin_qY,offset=target_offset)
                        updates=(
                            {name:neighbor_z,name+'_native':native['z_mean']},
                            {name:neighbor_thin,name+'_native':native_thin['z_mean']},
                            {name:decoder,name+'_native':decoder},
                            {name+'_native':native['entropy']})
                        coords.update(updates[0]); thincoords.update(updates[1])
                        decoders.update(updates[2]); entropy.update(updates[3])
                    except Exception as exc:
                        failures.append(dict(**key,regime=regime,method=name,error=str(exc)))
                        continue
                pred={m:decoders[m].predict(z) for m,z in coords.items()}
                thinpred={m:decoders[m].predict(z) for m,z in thincoords.items()}
                pred['quadratic_segment_coverage_oracle']=oracle_pred
                pred['segment_only_oracle']=segment_pred
                thinpred['quadratic_segment_coverage_oracle']=oracle_pred
                thinpred['segment_only_oracle']=segment_pred
                meta=dict(**key,regime=regime)
                scores.extend(primary_rows(eval_response,pred,variance,eval_a,meta))
                thin_scores.extend(primary_rows(eval_response,thinpred,variance,eval_a,meta))
                selected_scores.extend(selected_rows(eval_response,pred,response,sa,eval_a,panel,gene_to_j,meta))
                pooled=coords['pooled_scFates_neighbor']
                for method,z in coords.items():
                    move=np.abs(z-thincoords[method]); change=np.abs(z-pooled)
                    ent=entropy.get(method,np.full(len(z),np.nan))
                    for seg,label in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
                        use=np.ones(len(z),bool) if seg is None else eval_a==seg
                        if use.any():
                            stability.append(dict(**meta,method=method,segment=label,n=int(use.sum()),
                              quarter_read_mae=float(move[use].mean()),
                              fraction_read_moved_over_0_1=float(np.mean(move[use]>.1)),
                              coordinate_change_from_pooled_scFates=float(change[use].mean()),
                              rank_vs_pooled_scFates=float(spearmanr(z[use],pooled[use]).statistic),
                              native_entropy_mean=float(np.nanmean(ent[use])) if np.isfinite(ent[use]).any() else np.nan))
                    for sid,val,tval,lab,e in zip(eval_ids,z,thincoords[method],eval_a,ent):
                        positions.append(dict(**meta,method=method,structure_id=sid,z=float(val),
                          thinned_z=float(tval),segment=['S1','S2','S3'][int(lab)],
                          native_entropy=float(e) if np.isfinite(e) else np.nan))
                print('Completed:',source,'to',target,'fold',fold,regime,flush=True)
    frames=dict(scores=scores,selected_scores=selected_scores,thin_scores=thin_scores,
      stability=stability,positions=positions,diagnostics=diagnostics,joint_history=histories,
      failures=failures,sourcefolds=sourcefolds,pooled_graph=graphs)
    return {k:pd.DataFrame(v).to_json(orient='table') for k,v in frames.items()}


In [ ]:
CACHE_INPUTS=dict(expression=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,
 anatomy=anatomy,gene_fold=gene_fold,markers=marker_mask,n_spots=n_spots,panel=frozen_panel,
 reference_scores=reference_scores,reference_positions=reference_positions)
payload=cached_payload('groupwise_repeated_structure_reconstruction',run_groupwise,
 root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST)
outputs={k:pd.read_json(io.StringIO(str(v)),orient='table') for k,v in payload.items()}
for name,frame in outputs.items(): frame.to_csv(OUT/(name+'.csv'),index=False)

joint_methods={'joint_balanced_offset','joint_balanced_no_offset','joint_pooled_no_offset'}
attempt_keys=['source_specimen','test_specimen','fold','regime','method']
diag=outputs['diagnostics']
fails=outputs['failures']
joint_diag=diag[diag.method.isin(joint_methods)] if 'method' in diag else pd.DataFrame()
joint_fail=fails[fails.method.isin(joint_methods)] if 'method' in fails else pd.DataFrame()
returned=set(map(tuple,joint_diag[attempt_keys].itertuples(index=False,name=None))) if len(joint_diag) else set()
failed_keys=set(map(tuple,joint_fail[attempt_keys].itertuples(index=False,name=None))) if len(joint_fail) else set()
attempted=returned|failed_keys
nonconverged=int((~joint_diag.converged.fillna(False)).sum()) if len(joint_diag) else 0
converged=int(joint_diag.converged.fillna(False).sum()) if len(joint_diag) else 0
exception_count=int((~joint_fail.error.fillna('').str.startswith('nonconverged;')).sum()) if len(joint_fail) else 0
graph=outputs['pooled_graph']
graph_complete=graph.query('status=="complete"')
graph_regimes=graph_complete.groupby(['source_specimen','test_specimen','fold','regime']).method.nunique()
successful_graph_regimes=int((graph_regimes==2).sum())
graph_attempt_keys=set(map(tuple,graph[['source_specimen','test_specimen','fold','regime']].drop_duplicates().itertuples(index=False,name=None))) if len(graph) else set()
graph_failed_keys=set(map(tuple,fails.loc[fails.method=='pooled_graphs',
 ['source_specimen','test_specimen','fold','regime']].drop_duplicates().itertuples(index=False,name=None))) if len(fails) and 'method' in fails else set()
graph_attempt_keys |= graph_failed_keys
attempt_summary=pd.DataFrame([
 dict(analysis='joint_arms',planned_attempts=36,attempted_keys=len(attempted),
      returned_fits=len(returned),converged=converged,nonconverged=nonconverged,
      exceptions=exception_count,missing_attempts=max(36-len(attempted),0)),
 dict(analysis='pooled_graph_regimes',planned_attempts=12,attempted_keys=len(graph_attempt_keys),
      returned_fits=successful_graph_regimes,converged=successful_graph_regimes,
      nonconverged=0,exceptions=int((graph.status=='failed').sum()),
      missing_attempts=max(12-len(graph_attempt_keys),0))])
attempt_summary.to_csv(OUT/'model_attempt_summary.csv',index=False)
display(attempt_summary)

# Compare folds only on shared structure IDs; these ranges describe split sensitivity,
# not calibrated uncertainty. Pairwise Spearman uses each pair's own common IDs.
range_rows,pair_rows=[],[]
pos=outputs['positions']
fold_values=sorted(pos.fold.unique().tolist()) if len(pos) else [0,1,2]
group_keys=['source_specimen','test_specimen','regime','method','segment']
if len(pos):
    for group_key,frame in pos.groupby(group_keys,dropna=False):
        by_fold={int(f):g.set_index('structure_id').z for f,g in frame.groupby('fold')}
        missing=[f for f in [0,1,2] if f not in by_fold]
        id_union=set().union(*(set(v.index) for v in by_fold.values())) if by_fold else set()
        common_all=set.intersection(*(set(by_fold[f].index) for f in [0,1,2] if f in by_fold)) if by_fold else set()
        for sid in sorted(id_union):
            present=[f for f in [0,1,2] if f in by_fold and sid in by_fold[f].index]
            if len(present)>=2:
                values=np.array([by_fold[f].loc[sid] for f in present],float)
                sid_missing=[f for f in [0,1,2] if f not in present]
                range_rows.append(dict(zip(group_keys,group_key),structure_id=sid,folds=json.dumps(present),
                  missing_folds=json.dumps(sid_missing),fold_count=len(present),
                  common_structure_count=len(common_all),coordinate_min=float(values.min()),
                  coordinate_max=float(values.max()),coordinate_range=float(values.max()-values.min())))
        for fa,fb in [(0,1),(0,2),(1,2)]:
            common=sorted(set(by_fold.get(fa,pd.Series(dtype=float)).index)&
                          set(by_fold.get(fb,pd.Series(dtype=float)).index))
            rho=float(spearmanr(by_fold[fa].loc[common],by_fold[fb].loc[common]).statistic) if len(common)>=2 else np.nan
            pair_rows.append(dict(zip(group_keys,group_key),fold_a=fa,fold_b=fb,
              common_structure_count=len(common),fold_a_structure_count=len(by_fold.get(fa,[])),
              fold_b_structure_count=len(by_fold.get(fb,[])),missing_folds=json.dumps(missing),
              spearman=rho))
pd.DataFrame(range_rows).to_csv(OUT/'fold_coordinate_ranges.csv',index=False)
pd.DataFrame(pair_rows).to_csv(OUT/'fold_pairwise_agreement.csv',index=False)
if len(outputs['sourcefolds'].query('status=="complete"'))!=6:
    raise RuntimeError('Expected six completed source/fold reference fits; inspect failures.csv')
if len(outputs['pooled_graph'].query('status=="complete"'))!=24:
    raise RuntimeError('Expected 12 successful pooled DPT/scFates regimes; inspect failures.csv')
if not np.isfinite(outputs['positions'].z).all() or not outputs['positions'].z.between(0,1).all():
    raise ValueError('Invalid evaluation coordinates')
populations=outputs['positions'].groupby(['source_specimen','fold','regime','method']).structure_id.apply(lambda x:tuple(sorted(x)))
for (source,fold),group in populations.groupby(level=['source_specimen','fold']):
    expected=None
    for ids_for_arm in group:
        if expected is None: expected=ids_for_arm
        elif ids_for_arm!=expected: raise ValueError(f'Evaluation IDs differ: {source}/{fold}')
display(outputs['sourcefolds'])
display(outputs['pooled_graph'][['source_specimen','fold','regime','method','status']])
print('Joint fits converged:',converged,'of 36 planned attempts;',
      'exceptions:',exception_count,'failures:',len(outputs['failures']))
for name in ['scores','selected_scores']:
    frame=outputs[name]; keys=['source_specimen','fold','regime','segment']
    base=frame.query('method=="pooled_scFates_neighbor"')[keys+['normalized_mse']].rename(columns={'normalized_mse':'pooled_scFates_mse'})
    quad=frame.query('method=="quadratic_segment_coverage_oracle"')[keys+['normalized_mse']].rename(columns={'normalized_mse':'quadratic_oracle_mse'})
    seg=frame.query('method=="segment_only_oracle"')[keys+['normalized_mse']].rename(columns={'normalized_mse':'segment_only_mse'})
    comp=frame.merge(base,on=keys,validate='many_to_one').merge(quad,on=keys,validate='many_to_one').merge(seg,on=keys,validate='many_to_one')
    comp['gain_vs_pooled_scFates_percent']=100*(1-comp.normalized_mse/comp.pooled_scFates_mse)
    comp['gain_vs_quadratic_oracle_percent']=100*(1-comp.normalized_mse/comp.quadratic_oracle_mse)
    comp['gain_vs_segment_only_percent']=100*(1-comp.normalized_mse/comp.segment_only_mse)
    comp.to_csv(OUT/(name+'_comparison.csv'),index=False)
    summary=comp.groupby(['source_specimen','regime','segment','method'],dropna=False).agg(
      normalized_mse=('normalized_mse','mean'),fit_count=('fold','count'),fold_count=('fold','nunique'),
      gain_vs_pooled_scFates_percent=('gain_vs_pooled_scFates_percent','mean'),
      gain_vs_quadratic_oracle_percent=('gain_vs_quadratic_oracle_percent','mean'),
      gain_vs_segment_only_percent=('gain_vs_segment_only_percent','mean')).reset_index()
    summary.to_csv(OUT/(name+'_summary.csv'),index=False); display(summary)
stab=outputs['stability'].groupby(['source_specimen','regime','segment','method'],dropna=False).agg(
 quarter_read_mae=('quarter_read_mae','mean'),fraction_read_moved_over_0_1=('fraction_read_moved_over_0_1','mean'),
 coordinate_change_from_pooled_scFates=('coordinate_change_from_pooled_scFates','mean'),
 rank_vs_pooled_scFates=('rank_vs_pooled_scFates','mean'),native_entropy_mean=('native_entropy_mean','mean'),
 fit_count=('fold','count'),fold_count=('fold','nunique')).reset_index()
stab.to_csv(OUT/'stability_summary.csv',index=False); display(stab)


## Matched comparisons among accepted fits

After the first outputs, compare arms on the intersection of successful source/fold/regime cases. This is a reporting correction, not a new fit or query-selected parameter change. Unequal convergence sets otherwise make aggregate arm means inappropriate for ablations. Native/neighbor comparisons retain the same fitted atlas and decoder. Gene-panel agreement remains development sensitivity, not anatomical ground truth.

In [ ]:
pairs=[('joint_balanced_offset','joint_balanced_no_offset'),
 ('joint_balanced_no_offset','joint_pooled_no_offset'),
 ('joint_balanced_offset','source_scFates_neighbor'),
 ('joint_balanced_offset','pooled_scFates_neighbor'),
 ('pooled_scFates_neighbor','source_scFates_neighbor'),
 ('pooled_DPT_neighbor','source_DPT_neighbor'),
 ('joint_balanced_offset_native','joint_balanced_offset')]
match_keys=['source_specimen','test_specimen','fold','regime','segment']
for table_name,metric in [('scores','normalized_mse'),('selected_scores','normalized_mse'),
                          ('stability','quarter_read_mae')]:
    frame=outputs[table_name]
    parts=[]
    for candidate,baseline in pairs:
        left=frame[frame.method==candidate][match_keys+[metric]]
        right=frame[frame.method==baseline][match_keys+[metric]]
        matched=left.merge(right,on=match_keys,suffixes=('_candidate','_baseline'),validate='one_to_one')
        if matched.empty:
            continue
        matched['candidate']=candidate
        matched['baseline']=baseline
        matched['gain_percent']=100*(1-matched[metric+'_candidate']/matched[metric+'_baseline'])
        parts.append(matched)
    paired=pd.concat(parts,ignore_index=True)
    paired.to_csv(OUT/(table_name+'_matched_comparisons.csv'),index=False)
    matched_summary=paired.groupby(['source_specimen','regime','segment','candidate','baseline']).agg(
      gain_percent=('gain_percent','mean'),fit_count=('fold','count'),fold_count=('fold','nunique')).reset_index()
    matched_summary.to_csv(OUT/(table_name+'_matched_summary.csv'),index=False)
    display(matched_summary)


## Prediction and coverage robustness

Each plotted dot is a source-by-fold result, grouped by transfer direction and calibration regime. Tables preserve methods, segments, and both privileged oracle comparisons. Failed fits and nonconvergence remain in outputs and figure annotations; null and heterogeneous folds stay visible. Fold coordinate ranges are split-sensitivity summaries, not calibrated uncertainty. Pairwise fold correlations use only common structure IDs, and their row counts and missing folds are saved.


In [ ]:
import matplotlib.pyplot as plt
comparison=pd.read_csv(OUT/'selected_scores_comparison.csv')
methods=['pooled_DPT_neighbor','pooled_scFates_neighbor',
 'source_DPT_neighbor','source_scFates_neighbor','joint_balanced_offset',
 'joint_balanced_no_offset','joint_pooled_no_offset','joint_balanced_offset_native',
 'joint_balanced_no_offset_native','joint_pooled_no_offset_native']
colors={m:f'C{i}' for i,m in enumerate(methods)}
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True)
for col,source in enumerate(CONTROLS):
 for row,regime in enumerate(['full','S3_only']):
  ax=axes[row,col]
  for method in methods:
   local=comparison.query('source_specimen==@source and regime==@regime and method==@method')
   for i,segment in enumerate(['S1_selected','S2_selected','S3_selected']):
    values=local.loc[local.segment==segment,'gain_vs_pooled_scFates_percent'].to_numpy()
    if len(values):
     x=i+(methods.index(method)-(len(methods)-1)/2)*.055
     ax.scatter(np.full(len(values),x),values,color=colors[method],s=18,label=method if i==0 else None)
     ax.plot([x-.025,x+.025],[values.mean()]*2,color=colors[method],linewidth=1)
  ax.axhline(0,color='gray',linewidth=.7)
  ax.set(xticks=[0,1,2],xticklabels=['S1','S2','S3'],title=f'Source {source} / {regime}',
         ylabel='Gain vs pooled scFates (%)')
handles,legend_labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,legend_labels,loc='lower center',frameon=False,fontsize=7,ncol=3)
failed=len(outputs['failures'])
fig.suptitle(f'Frozen source decoders; disjoint evaluation; failures={failed}; nonconverged={nonconverged}')
fig.tight_layout(rect=[0,.12,1,.95]); fig.savefig(OUT/'selected_program_gains.png',dpi=160,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(9,5))
stab=outputs['stability']
for method in methods:
 local=stab.query('segment=="all" and method==@method')
 if len(local): ax.scatter(local.quarter_read_mae,local.coordinate_change_from_pooled_scFates,
                           label=method,alpha=.8)
ax.set(xlabel='Quarter-read coordinate movement (MAE)',
       ylabel='Coordinate change from pooled scFates',
       title='Paired query thinning with frozen fits')
ax.legend(frameon=False,fontsize=8,ncol=2)
fig.tight_layout(); fig.savefig(OUT/'quarter_read_stability.png',dpi=160); plt.show()


## Readout

Interpret transfer directions separately and retain null or heterogeneous folds. A model that changes coordinates without improving within-segment prediction or read robustness has not shown a useful repeated-structure contribution. Query anatomy enters only stress-population selection, privileged oracle prediction, stratified scoring, and summaries. No outputs from this notebook were run during authoring.


Measured result: 21 of 36 joint fits converged (14/18 full calibration and 7/18 S3-only); all pooled graph comparisons and six source references completed. The repeated Gaussian atlas does not establish a consistent gain over source-only scFates. On matching accepted full-calibration folds, balanced-offset selected-program gains versus that reference are −3.37%/+8.46%/−9.78% (S1/S2/S3, Ctrl1A2 source, two folds) and +0.16%/−4.84%/+0.57% (reverse, three). Read movement is greater. Offsets add negligible benefit and weighting comparisons have limited shared convergence sets.

The important design evidence is that adding calibration profiles can degrade fine gene-panel ordering, especially for pooled scFates. Continuous source references still predict these development-selected programs better than segment-only and coverage comparators. A shared reconstruction must preserve reproducible local organization; neither a smooth pooled curve nor gains against an unstable comparator proves that. This prototype is not adopted. The ledger records a source-defined regularization-unit check before another biological fit, as well as uniform-mixture and anatomical-identifiability limitations.
